In [ ]:
"""
Student Risk Early-Warning System — Gradio prototype
Ashesi University · Intro to AI · Group 5

STATUS: Running in DEMO MODE with placeholder prediction logic.
When Sprint 2's tuned model is finalized, replace compute() with
the "REAL MODEL VERSION" shown in the comment block near the bottom —
nothing else in this file needs to change.
"""

import random
import pandas as pd
import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
#  PREDICTION LOGIC — this is the ONE section to swap later
# ============================================================

def compute(school, sex, age, address, famsize, Pstatus, Medu, Fedu, Mjob, Fjob,
            reason, guardian, traveltime, studytime, failures, schoolsup, famsup,
            paid, activities, nursery, higher, internet, romantic, famrel, freetime,
            goout, Dalc, Walc, health, absences):
    """
    PLACEHOLDER — demo mode only.
    Approximates plausible model behavior using known-important features
    (failures, absences, study time, support) so the interface is fully
    testable and presentable while the real model is still being tuned.
    Not a trained model. Swap for the REAL MODEL VERSION below when ready.
    Returns (probs_dict, reasons_markdown) — this is the single contract
    the rest of the app depends on. Keep that return shape when you swap it.
    """
    score = 0.0
    score += (failures / 3) * 0.34
    score += (min(absences, 24) / 24) * 0.22
    score += ((4 - studytime) / 3) * 0.16
    score += (goout / 5) * 0.06
    score += ((6 - health) / 5) * 0.05
    score += (Walc / 5) * 0.05
    score += 0.06 if schoolsup == "no" else 0
    score += 0.06 if famsup == "no" else 0
    score += random.uniform(-0.04, 0.04)  # small jitter so repeat clicks aren't identical
    score = max(0.02, min(0.95, score))

    high = score
    low = max(0.02, (1 - score) * 0.55)
    medium = max(0.02, 1 - high - low)
    total = high + medium + low
    probs = {"High": high / total, "Medium": medium / total, "Low": low / total}

    contributing = []
    if failures > 0:
        contributing.append(f"- **{failures} past failure{'s' if failures > 1 else ''}** — academic advising referral")
    if absences >= 10:
        contributing.append(f"- **{absences} absences** — attendance check-in")
    if studytime <= 1:
        contributing.append("- **Low weekly study time** — study-skills workshop")
    if schoolsup == "no":
        contributing.append("- **No school academic support** — connect with tutoring center")
    if famsup == "no":
        contributing.append("- **No family academic support** — peer mentoring program")

    reasons_md = "**Top contributing factors:**\n" + "\n".join(contributing[:3]) if contributing \
        else "No strongly elevated risk factors detected for this student."

    demo_notice = "\n\n*(Demo mode — placeholder logic, not the trained model)*"
    return probs, reasons_md + demo_notice


def make_gauge(probs):
    """Donut chart of the three risk probabilities — visual companion to gr.Label."""
    plt.close("all")  # prevent figure accumulation across repeated calls
    colors = {"High": "#B23A2C", "Medium": "#B87A1A", "Low": "#3C7A52"}
    labels = list(probs.keys())
    sizes = [probs[l] for l in labels]
    wedge_colors = [colors[l] for l in labels]

    fig, ax = plt.subplots(figsize=(3.4, 3.4))
    ax.pie(sizes, colors=wedge_colors, startangle=90,
           wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2))
    top_label = max(probs, key=probs.get)
    ax.text(0, 0.12, top_label, ha="center", va="center", fontsize=16,
            fontweight="bold", color=colors[top_label])
    ax.text(0, -0.15, f"{probs[top_label]*100:.0f}% likely", ha="center", va="center",
            fontsize=9.5, color="#6B5E52")
    ax.set(aspect="equal")
    fig.patch.set_alpha(0)
    plt.tight_layout()
    return fig


def live_predict(*args):
    """Fires on every input change — instant feedback, nothing logged to history."""
    probs, reasons_md = compute(*args)
    fig = make_gauge(probs)
    return probs, reasons_md, fig


def assess_and_log(*args_and_history):
    """Fires on the Assess Risk button — computes AND records the profile to history."""
    *args, history = args_and_history
    probs, reasons_md = compute(*args)
    fig = make_gauge(probs)
    top = max(probs, key=probs.get)
    row = {
        "#": len(history) + 1,
        "Failures": args[14],
        "Absences": args[29],
        "Study Time": args[13],
        "Predicted Risk": top,
        "Confidence": f"{probs[top]*100:.0f}%",
    }
    history = history + [row]
    df = pd.DataFrame(history)
    return probs, reasons_md, fig, history, df


# ============================================================
#  THEME — Ashesi maroon accent on a clean light theme
# ============================================================

theme = gr.themes.Soft(
    primary_hue="red",
    secondary_hue="orange",
    neutral_hue="stone",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
    font_mono=[gr.themes.GoogleFont("IBM Plex Mono"), "monospace"],
).set(
    button_primary_background_fill="#8A2E22",
    button_primary_background_fill_hover="#5E1F17",
    button_primary_text_color="#FFFFFF",
    block_title_text_color="#5E1F17",
    block_label_text_color="#6B5E52",
)

CSS = """
#header_md h1 { margin-bottom: 2px; }
#banner { background: #F6EBD8; border: 1px solid #E3C6BC; border-radius: 8px; padding: 10px 14px; }
"""

# ============================================================
#  LAYOUT
# ============================================================

with gr.Blocks(title="Student Risk Early-Warning System") as demo:

    gr.Markdown(
        "# Student Risk Early-Warning System\n"
        "Ashesi University · Intro to AI · Group 5",
        elem_id="header_md",
    )
    gr.Markdown(
        "⚠️ **Demo mode** — this build uses placeholder logic while the trained model is finalized. "
        "Predictions below are illustrative, not live model output.",
        elem_id="banner",
    )

    with gr.Row():
        with gr.Column(scale=3):
            with gr.Tab("Personal & Family"):
                school = gr.Dropdown(["GP", "MS"], value="GP", label="School")
                sex = gr.Dropdown(["F", "M"], value="F", label="Sex")
                age = gr.Slider(15, 22, value=17, step=1, label="Age")
                address = gr.Dropdown(["U", "R"], value="U", label="Home address")
                famsize = gr.Dropdown(["LE3", "GT3"], value="GT3", label="Family size")
                Pstatus = gr.Dropdown(["T", "A"], value="T", label="Parents' cohabitation")
                Medu = gr.Slider(0, 4, value=2, step=1, label="Mother's education")
                Fedu = gr.Slider(0, 4, value=2, step=1, label="Father's education")
                Mjob = gr.Dropdown(["teacher", "health", "services", "at_home", "other"], value="other", label="Mother's job")
                Fjob = gr.Dropdown(["teacher", "health", "services", "at_home", "other"], value="other", label="Father's job")
                guardian = gr.Dropdown(["mother", "father", "other"], value="mother", label="Guardian")

            with gr.Tab("School & Support"):
                reason = gr.Dropdown(["home", "reputation", "course", "other"], value="course", label="Reason for choosing school")
                traveltime = gr.Slider(1, 4, value=1, step=1, label="Travel time to school")
                studytime = gr.Slider(1, 4, value=2, step=1, label="Weekly study time")
                failures = gr.Slider(0, 3, value=0, step=1, label="Past class failures")
                schoolsup = gr.Dropdown(["yes", "no"], value="no", label="Extra school support")
                famsup = gr.Dropdown(["yes", "no"], value="yes", label="Family academic support")
                paid = gr.Dropdown(["yes", "no"], value="no", label="Extra paid classes")
                activities = gr.Dropdown(["yes", "no"], value="yes", label="Extra-curricular activities")
                nursery = gr.Dropdown(["yes", "no"], value="yes", label="Attended nursery school")
                higher = gr.Dropdown(["yes", "no"], value="yes", label="Wants higher education")
                internet = gr.Dropdown(["yes", "no"], value="yes", label="Internet access at home")

            with gr.Tab("Behavior & Wellbeing"):
                romantic = gr.Dropdown(["yes", "no"], value="no", label="In a relationship")
                famrel = gr.Slider(1, 5, value=4, step=1, label="Family relationship quality")
                freetime = gr.Slider(1, 5, value=3, step=1, label="Free time after school")
                goout = gr.Slider(1, 5, value=3, step=1, label="Going out with friends")
                Dalc = gr.Slider(1, 5, value=1, step=1, label="Workday alcohol use")
                Walc = gr.Slider(1, 5, value=1, step=1, label="Weekend alcohol use")
                health = gr.Slider(1, 5, value=4, step=1, label="Current health status")
                absences = gr.Slider(0, 40, value=4, step=1, label="Absences this term")

            with gr.Row():
                predict_btn = gr.Button("Assess Risk", variant="primary", scale=2)
                clear_btn = gr.ClearButton(scale=1)

        with gr.Column(scale=2):
            gr.Markdown("### Result — updates live as you adjust indicators")
            gauge_plot = gr.Plot(label="Risk Breakdown")
            risk_output = gr.Label(label="Predicted Risk Level")
            reasons_output = gr.Markdown()

    all_inputs = [school, sex, age, address, famsize, Pstatus, Medu, Fedu, Mjob, Fjob,
                  reason, guardian, traveltime, studytime, failures, schoolsup, famsup,
                  paid, activities, nursery, higher, internet, romantic, famrel, freetime,
                  goout, Dalc, Walc, health, absences]

    gr.Examples(
        examples=[
            # school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,
            # traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,
            # internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
            ["GP", "F", 17, "U", "GT3", "T", 1, 1, "at_home", "other", "course", "mother",
             2, 1, 2, "no", "no", "no", "no", "yes", "no", "no", "no", 3, 4, 4, 3, 4, 3, 20],
            ["MS", "M", 16, "U", "LE3", "T", 4, 4, "health", "teacher", "reputation", "father",
             1, 4, 0, "yes", "yes", "yes", "yes", "yes", "yes", "yes", "no", 5, 2, 1, 1, 1, 5, 0],
            ["GP", "M", 17, "R", "GT3", "T", 2, 2, "services", "services", "home", "mother",
             2, 2, 0, "no", "yes", "no", "yes", "yes", "yes", "yes", "yes", 3, 3, 3, 2, 3, 3, 8],
        ],
        inputs=all_inputs,
        outputs=[risk_output, reasons_output, gauge_plot],
        fn=live_predict,
        label="Try a preset student — click to load and instantly assess",
        cache_examples=False,
    )

    gr.Markdown("### Assessment History")
    history_state = gr.State([])
    history_table = gr.Dataframe(
        headers=["#", "Failures", "Absences", "Study Time", "Predicted Risk", "Confidence"],
        label="Every profile assessed with the button, in this session",
        interactive=False,
    )

    # Live preview: any input change updates the result instantly, nothing logged
    for inp in all_inputs:
        inp.change(fn=live_predict, inputs=all_inputs, outputs=[risk_output, reasons_output, gauge_plot])

    # Explicit action: the button both shows the result AND records it to history
    predict_btn.click(
        fn=assess_and_log,
        inputs=all_inputs + [history_state],
        outputs=[risk_output, reasons_output, gauge_plot, history_state, history_table],
    )

    clear_btn.add(all_inputs + [risk_output, reasons_output, gauge_plot])

if __name__ == "__main__":
    demo.launch(share=True, theme=theme, css=CSS)


# ============================================================
#  REAL MODEL VERSION — paste this over compute() above
#  once student_risk_model.pkl is ready. Nothing else in this
#  file needs to change — live_predict(), assess_and_log(), the
#  gauge, the history table, and the examples all call compute()
#  and only care about its (probs_dict, reasons_markdown) return
#  shape, not what's inside it.
# ============================================================
#
# import joblib
# bundle = joblib.load("student_risk_model.pkl")
# model = bundle["model"]
# label_encoders = bundle["label_encoders"]
# onehot_encoder = bundle["onehot_encoder"]
# scaler = bundle["scaler"]
# binary_features = bundle["binary_features"]
# categorical_features = bundle["categorical_features"]
# numeric_features = bundle["numeric_features"]
#
# importances = pd.Series(model.feature_importances_, index=model.feature_names_in_)
# direction = {"failures": 1, "absences": 1, "studytime": -1, "famrel": -1,
#              "goout": 1, "Dalc": 1, "Walc": 1, "health": -1}
# labels_map = {"failures": "past class failures", "absences": "absences",
#               "studytime": "low study time", "famrel": "weaker family relationships",
#               "goout": "frequent socializing", "Dalc": "weekday alcohol use",
#               "Walc": "weekend alcohol use", "health": "lower health rating"}
#
# def compute(school, sex, age, address, famsize, Pstatus, Medu, Fedu, Mjob, Fjob,
#             reason, guardian, traveltime, studytime, failures, schoolsup, famsup,
#             paid, activities, nursery, higher, internet, romantic, famrel, freetime,
#             goout, Dalc, Walc, health, absences):
#     raw = pd.DataFrame([{
#         "school": school, "sex": sex, "age": age, "address": address, "famsize": famsize,
#         "Pstatus": Pstatus, "Medu": Medu, "Fedu": Fedu, "Mjob": Mjob, "Fjob": Fjob,
#         "reason": reason, "guardian": guardian, "traveltime": traveltime, "studytime": studytime,
#         "failures": failures, "schoolsup": schoolsup, "famsup": famsup, "paid": paid,
#         "activities": activities, "nursery": nursery, "higher": higher, "internet": internet,
#         "romantic": romantic, "famrel": famrel, "freetime": freetime, "goout": goout,
#         "Dalc": Dalc, "Walc": Walc, "health": health, "absences": absences,
#     }])
#     bin_df = pd.DataFrame(index=raw.index)
#     for col in binary_features:
#         bin_df[col] = label_encoders[col].transform(raw[col])
#     cat_arr = onehot_encoder.transform(raw[categorical_features])
#     cat_df = pd.DataFrame(cat_arr, columns=onehot_encoder.get_feature_names_out(categorical_features), index=raw.index)
#     num_arr = scaler.transform(raw[numeric_features])
#     num_df = pd.DataFrame(num_arr, columns=numeric_features, index=raw.index)
#     X_input = pd.concat([bin_df, cat_df, num_df], axis=1)
#     X_input = X_input[model.feature_names_in_]
#     probs = model.predict_proba(X_input)[0]
#     prob_dict = dict(zip(model.classes_, probs))
#
#     contributions = []
#     for feat, sign in direction.items():
#         elevated = num_df[feat].values[0] * sign
#         if elevated > 0.4:
#             contributions.append((feat, importances.get(feat, 0) * elevated))
#     contributions.sort(key=lambda x: x[1], reverse=True)
#     reasons_md = "**Top contributing factors:**\n" + "\n".join(
#         f"- {labels_map[f]}" for f, _ in contributions[:3]
#     ) if contributions else "No strongly elevated risk factors detected for this student."
#
#     return prob_dict, reasons_md

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8597beb471a92cf3ff.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
